# 3.2 • Compare representations fairly

**Learning objectives:** align observations by ID; hold cohort and split constant; distinguish access constraints from predictive quality.

**How to learn:** predict → run → explain → change one thing. Spend 45–90 minutes here.

**Data:** the default worked example is deterministic synthetic practice data near Colorado. It is not SNOTEL, a watershed survey, a burn map, or a foundation-model output.


## Before you run

A fair comparison changes the representation while retaining labels, point locations, year, classifier and held-out groups. Missing coverage can change the population. Compare on the common cohort and report all exclusions. More vector dimensions do not guarantee greater accuracy.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course import OUT, points


## Same points, different simulated representations


In [ ]:
from course import embeddings,evaluate
frame=points()
a=embeddings(frame,64,12)
b=embeddings(frame,128,13)
ma,sa,ca,ta,va=evaluate(frame,a)
mb,sb,cb,tb,vb=evaluate(frame,b)
np.testing.assert_array_equal(ta,tb);np.testing.assert_array_equal(va,vb)
comparison=pd.DataFrame([sa,sb],index=['synthetic-64','synthetic-128'])
print(comparison)
comparison.to_csv(OUT/'synthetic_comparison.csv')
comparison[['balanced_accuracy','baseline']].plot.bar(ylim=(0,1),rot=0);plt.show()


## Missing rows: align by keys, never by position


In [ ]:
left=pd.DataFrame(a,index=frame.point_id)
right=pd.DataFrame(b,index=frame.point_id).sample(frac=.9,random_state=42)
common=frame[frame.point_id.isin(right.index)].reset_index(drop=True)
aa=left.loc[common.point_id].to_numpy()
bb=right.loc[common.point_id].to_numpy()
assert len(aa)==len(bb)==len(common)
print('Excluded from comparison:',len(frame)-len(common))
print(evaluate(common,aa)[1],evaluate(common,bb)[1])


## Your turn

Shuffle the right-hand table and demonstrate why matching by row number breaks the experiment. Write three criteria other than accuracy that matter when choosing an embedding product.

Use a new cell below to try this before opening the solution.


In [ ]:
# Write your experiment here. The worked example above is already complete.


<details><summary>Solution and reasoning — open after trying</summary>

`right.sample(frac=1)` changes order; `.loc[common.point_id]` restores alignment. Compare geographic/year coverage, reproducibility/versioning, access terms, download size, and model transparency. Open data, open code and open weights are separate claims.

</details>


## Check yourself

Expected: equal test indices for aligned representations and 24 excluded points in the missing-data example. Can these synthetic scores establish that AlphaEarth or TESSERA is better? No.

**Exit ticket:** write one result, one explanation, and one limitation. Save the notebook with your own observations.


## Transfer to real data

Use `scripts/fetch_embeddings.py tessera` with the same labels and year as AlphaEarth. The script samples native pixels through each tile CRS/transform and keeps unavailable values missing. `scripts/compare_embeddings.py` checks metadata, aligns IDs, removes invalid rows jointly and evaluates the same cohort. Record actual coverage and product version; do not infer it from an old global-coverage claim.
